# Simple LLM App

In [ ]:
## A brach new ChatGPT app

There is a lot of buzz aroung Generative AIs and ChatGpt, and your boss asked for its own ChatGPT.

Tou have found some straigthforward examples at the Streamlit and Langchain sites. So let's go!

LangChain is a new open source framework that allows AI developers to combine Large Language Models (LLMs) with external data. It also provides a lot of helpers for connectivity with sources and vector databases.

LangChain resources
> - Landpage: https://readthedocs.org/projects/langchain/db2d
> - Comonents: https://docs.langchain.com/docs/category/components
> - git: https://github.com/hwchase17/langchain.git
> - API Reference: https://api.python.langchain.com/en/latest/

Let's start with the [basic LLM app in 18 lines of code from the 
streamlit documentation](https://blog.streamlit.io/langchain-tutorial-1-build-an-llm-powered-app-in-18-lines-of-code/)

Here is the full app code. It is really minimalistic, a text box for the question, the response is displayed right under.

```python
import streamlit as st
from langchain.llms import OpenAI

st.title('🦜🔗 Quickstart App')

openai_api_key = st.sidebar.text_input('OpenAI API Key')

def generate_response(input_text):
  llm = OpenAI(temperature=0.7, openai_api_key=openai_api_key)
  st.info(llm(input_text))

with st.form('my_form'):
  text = st.text_area('Enter text:', 'What are the three key pieces of advice for learning how to code?')
  submitted = st.form_submit_button('Submit')
  if not openai_api_key.startswith('sk-'):
    st.warning('Please enter your OpenAI API key!', icon='⚠')
  if submitted and openai_api_key.startswith('sk-'):
    generate_response(text)
```

## The day after

After a short night, populated with FinOps guys with black suits, black sun glasses, pointing at you a menacing red pen throwing bad figures, you wake up with an idea. What if using a mock for dev tests?

Let's check whether we can replace the OpenAI model with some strings in order to test the GUI.

The interaction with the model is located in the function __generate_response__. Does not seem to hard to workaround.

```python
def generate_response(input_text):
  llm = OpenAI(temperature=0.7, openai_api_key=openai_api_key)
  st.info(llm(input_text))
```

As st.write expect a string, the llm() output should be a string.

As a first attempt, we could replace the llm part with some fixed text. 

Let's create 2 functions for query answering, one calls OpenAI and the other one is a faker.

For instance:
    
```python
def query_llm(input_text):
    llm = OpenAI(temperature=0.7, openai_api_key=openai_api_key)
    return llm(input_text)

def query_llm_fake(input_text):
    response = "384,400 km"
    return response


Sounds great. Now test the OpenAI version.

```python
def generate_response(input_text):
    response = query_llm(input_text)
    st.info(response)
```

To the question "What is the distance to the Moon?", we get the right answer:
> "The average distance from Earth to the Moon is 238,855 miles (384,400 kilometers)".

And to the question  "Who is the White Rabbit?", we get a typical ChatGPT answer:
> The White Rabbit is a fictional character from the book Alice's Adventures in Wonderland by Lewis Carroll. He appears at the very beginning of the story, in which Alice follows him down a rabbit hole, and is noted for his continual use of the phrase "Oh dear! Oh dear! I shall be too late!" He is often referred to as the March Hare, due to his association with the March Hare in the book's sequel, Through the Looking-Glass.



Now switch  to the fake model.

```python
def generate_response(input_text):
    #response = query_llm(input_text)
    response = query_llm_fake(input_text)
    st.info(response)
```

Well, it works fine, though it is somewhat limited. 

It always gives the same answer which makes testing the GUI a little boring.

## A better option

The first test made things clear. 
I would like to have at least a series of predefined responses.

I'm not alone. This is the goal of the Fake LLM provided by Langchain.

Let's try the code below which leverages the fake LLM.

Write the class, feed the faker with some responses usefull to size the GUI elements, check the rendering when testing the GUI.

```python
def query_llm_fake(input_text):
    fake_responses = [
        "384,400 km",
        "The White Rabbit is a character of Alice of Wonderland and is always late"
    ]
    llm = FakeListLLM(responses=fake_responses) 
    response = llm(input_text)
    return response
```

First enter  "Waht is the distance to the Moon?"

It diaplys "384,400 km", Great!

Then try "Who is the White Rabbit?"

And it displays ... "384,400 km" ..., Hmmm.

What went wrong?

Moving helps solving problems in the background. 

After a jump to the coffee machine, I featured how this thing works.

The fake LLM reads a sequence of strings (e.g list of response) in order to serve them one by one.
Each time the fake LLM is called, it yields the next answer. 
But as the Fake LMM is reset each call, it always yields the firt answer.

Let's fix it by keeping track of the llm instance inside the function.

```python
def query_llm_fake(input_text):
    # using a static variable local to the function
    query_llm_fake.llm = getattr(query_llm_fake, 'llm', None) 
    if not query_llm_fake.llm:
        # initialize only of not set yet
        fake_responses = [
            "384,400 km",
            "The White Rabbit is a character of Alice of Wonderland and is always late"
        ]
        query_llm_fake.llm = FakeListLLM(responses=fake_responses) 
    response = query_llm_fake.llm(input_text)
    return response
```

Now the app correctly displays "384,400 km" on first query  and  "The White Rabbit is a character of Alice of Wonderland and is always late" on the second query.

Please note that answers will always come in this order whatever the questions are.

In addition, trying to query the model a third time results in a list index out of range error because it attempts to read a third answer from a two elements list.

Hmmm, need more answers.

## With a little hep from the NLP dataset

Hopefully NLP benchmarks have a similar need for large sets of answers and they curated datasets.

Squad is a Question and answer dataset available as a JSON file. 
> - SQUAD page https://rajpurkar.github.io/SQuAD-explorer/
> - SQUAD Dataset https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json
> - other datasets https://paperswithcode.com/task/question-answering#:~:text=Popular%20benchmark%20datasets%20for%20evaluation%20question%20answering%20systems%20include%20SQuAD,models%20are%20T5%20and%20XLNet.

The file structure is quite complex, however questions and answers are easy to find.
Some elements to take into considération:
    - The file has multiple areas. 
    - In each area, there are paragraphs consisting in a context, questions and answers.
    - Most questions have multiple answers. However some are empty. 
    - Each answer tracks the start of the text occurence in the context.

So now let's rework the faker in order to give more answers.

Here is an example of a faker based on the QA dataset. 
The fake LMM initializes itself with a random sampling of Squad answers.
Then it behaves like the FakeListLLM.

```python
from langchain.llms.fake import FakeListLLM
import urllib.request
import json 
from random import randrange

class RandomQAFakeLLM(FakeListLLM):
    def __init__(self, size):
        data = self._load_data()
        responses = self._sample(data, size)
        super().__init__(responses=responses)
        
    def _load_data(self):
        # loads the dataset
        squad_dataset_path = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json"
        with urllib.request.urlopen(squad_dataset_path) as url:
            data= json.load(url)
        return data

    def _sample(self, data, size):
        # randomly pick answers 
        answers = []
        while len(answers) < size:
            a = randrange(len(data['data']))
            p = randrange(len(data['data'][a]['paragraphs']))
            q = randrange(len(data['data'][a]['paragraphs'][p]['qas']))
            nr_t = len(data['data'][a]['paragraphs'][p]['qas'][q]['answers'])
            if nr_t > 0:
                t = randrange(nr_t)
                answer = data['data'][a]['paragraphs'][p]['qas'][q]['answers'][0]['text']
                answers.append(answer)
        return answers


def query_llm_fake(input_text):
    # using a static variable local to the function
    query_llm_fake.llm = getattr(query_llm_fake, 'llm', None) 
    if not query_llm_fake.llm:
        query_llm_fake.llm = RandomQAFakeLLM(20) 
    response = query_llm_fake.llm(input_text)
    return response
```

Now the app wil get answers like below to whatever the question typed in as input.
- Western art from the Middle Ages to the present
- Washington and Thomas Gage
- the 1970s


## Concluson

This option is for tests purpose only.
The fake llm s not clever at all.

However it is usefull to mimic the real model behavior and putputs for dev purpose.

There are advantages during the development phase:
- They are free. you don't have to pay for the rtrial and error steps.
- The response is quite immediate while testing with real models requires API calls and may suffer fromlatency.
- The model does not requires credentials and secret management
- The response can be very deterministic if the random list make use of a seed

They are a good fit for unit testing. Will see this in the next post.


# Material

## Initializations

In [ ]:
### Update environment

In [ ]:
!apt-get update && apt-get install -y build-essential 1>/dev/null

In [ ]:
!apt-get update && apt-get install -y jq 1>/dev/null

In [ ]:
!pip install --upgrade pip  1>/dev/null

## Requirements

In [ ]:
!pip install langchain==0.0.230 1>/dev/null

In [ ]:
!pip install openai==0.27.8 1>/dev/null

## Secrets and credentials

In [ ]:
%%bash --out secrets 
# using AWS's Secret Manager to store keys
# garb the keys and store it into a Pytthon variable
export RESPONSE=$(aws secretsmanager get-secret-value --secret-id 'salvia/labbench/tests' )
export SECRETS=$( echo $RESPONSE | jq '.SecretString | fromjson')

echo $SECRETS

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = eval(secrets)["OPENAI_API_KEY"]


## version 1

change in generate_response

```python
def get_llm_model():
    llm = OpenAI(temperature=0.7, openai_api_key=openai_api_key)
    return llm(input_text)

def generate_response(input_text):
    llm = get_llm_model()
    st.info(llm(input_text))
```

In [ ]:
import os
from langchain.llms import OpenAI
from langchain.llms.fake import FakeListLLM

openai_api_key = os.environ["OPENAI_API_KEY"] 

def get_llm_model():
    llm = OpenAI(temperature=0.7, openai_api_key=openai_api_key)
    return llm

def get_fake_llm_model():
    fake_responses = [
        "384,400 km",
        "The White Rabbit is a character of Alice of Wonderland and is always late"
    ]
    llm = FakeListLLM(responses=fake_responses) 
    return llm


In [ ]:
fake = False
llm = get_fake_llm_model() if fake else get_llm_model()
    
query = "What is the distance to the Moon?"
print(llm(query))

query = "Who is the White Rabbit?"
print(llm(query))

In [ ]:
fake = True
llm = get_fake_llm_model() if fake else get_mlm_model()
    
query = "What is the distance to the Moon?"
print(llm(query))

query = "Who is the White Rabbit?"
print(llm(query))


In [ ]:
# result in list index out of range if an extra query is placed

In [ ]:

query = "What is the Capital of France?"
print(llm(query))


## Fake LLM version 2

## QA dataset

In [ ]:
import urllib.request
import json 
from pprint import pprint

squad_dataset_path = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json"
with urllib.request.urlopen(squad_dataset_path) as url:
    data= json.load(url)

print(f"root level attributes {data.keys()}")
print(f"number of areas {len(data['data'])}")
print(f"area attributes {data['data'][0].keys()}")
print(f"paragraphs attributes {data['data'][0]['paragraphs'][0].keys()}")
print(f"qas attributes {data['data'][0]['paragraphs'][0]['qas'][0].keys()}")
print(f"question sample {data['data'][0]['paragraphs'][0]['qas'][0]['question']}")
print(f"answers attributes {data['data'][0]['paragraphs'][0]['qas'][0]['answers'][0].keys()}")
print(f"answers attributes {data['data'][0]['paragraphs'][0]['qas'][0]['answers'][0]['text']}")


In [ ]:
# randomly pick answers
from random import randrange
from pprint import pprint

answers = []
while len(answers) < 20:
    a = randrange(len(data['data']))
    p = randrange(len(data['data'][a]['paragraphs']))
    q = randrange(len(data['data'][a]['paragraphs'][p]['qas']))
    nr_t = len(data['data'][a]['paragraphs'][p]['qas'][q]['answers'])
    if nr_t > 0:
        t = randrange(nr_t)
        answer = data['data'][a]['paragraphs'][p]['qas'][q]['answers'][0]['text']
        answers.append(answer)

pprint(answers)

## Fake LLM version 3

In [ ]:
## Fake LLM version 4

In [ ]:
from langchain.llms.fake import FakeListLLM
import urllib.request
import json 
from random import randrange

class RandomQAFakeLLM(FakeListLLM):
    def __init__(self, size):
        data = self._load_data()
        responses = self._sample(data, size)
        super().__init__(responses=responses)
        
    def _load_data(self):
       # loads the dataset
        squad_dataset_path = "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json"
        with urllib.request.urlopen(squad_dataset_path) as url:
            data= json.load(url)
        return data

    def _sample(self, data, size):
        # randomly pick answers
        answers = []
        while len(answers) < size:
            a = randrange(len(data['data']))
            p = randrange(len(data['data'][a]['paragraphs']))
            q = randrange(len(data['data'][a]['paragraphs'][p]['qas']))
            nr_t = len(data['data'][a]['paragraphs'][p]['qas'][q]['answers'])
            if nr_t > 0:
                t = randrange(nr_t)
                answer = data['data'][a]['paragraphs'][p]['qas'][q]['answers'][0]['text']
                answers.append(answer)
        return answers


def query_llm_fake(input_text):
    # using a static variable local to the function
    query_llm_fake.llm = getattr(query_llm_fake, 'llm', None) 
    if not query_llm_fake.llm:
        query_llm_fake.llm = RandomQAFakeLLM(20) 
    response = query_llm_fake.llm(input_text)
    return response

query = "What is the distance to the Moon?"
print(query_llm_fake(query))

query = "Who is the White Rabbit?"
print(query_llm_fake(query))

query = "What is the Capital of France?"
print(query_llm_fake(query))


In [ ]:
!mkdir -p work/llmops/simpleLLMApp

In [ ]:
%%writefile work/llmops/simpleqa/streamlit_app.py

import streamlit as st

# create a simple page
st.title('Simple QA')

In [ ]:
Now point your browser to one of the urls.  

# Evaluation of a query

In [ ]:
# TODO 2 QAstrategies + 2 lists as fakes

In [ ]:
class LLMStrategy:
    def __init__(self):
        self.model = None
        
    def get_model(self):
        return self.model

In [ ]:
from langchain.llms.fake import FakeListLLM

class FakeLLMStrategy(LLMStrategy):
    def __init__(self, responses):
        self.responses = responses
        self.model = None
        
    def get_model(self):
        self.model = FakeListLLM(responses=self.responses)
        return self.model


In [ ]:
from langchain.llms import OpenAI

class OpenAILLMStrategy(LLMStrategy):
    def __init__(self, llm_parameters):
        self.model = OpenAI(temperature=llm_parameters['temperature'], 
                             model_name=llm_parameters['model_name'])
 

In [ ]:
llm_parameters = {
    # Note, the default model is already 'text-davinci-003' 
    'model_name': "text-davinci-003",
    # temperature 0 means no randomness
    'temperature': 0
}

In [ ]:
fake_responses = ["White Rabbit is always late"]

In [ ]:
from langchain.chains import RetrievalQA

llm_service = FakeLLMStrategy(fake_responses)
#llm_sergice = OpenAILLMStrategy(llm_parameters)

llm_model = llm_service.get_model()

# Asking theLLM
# the response will be based on the retrieved documents 
qa_chain = RetrievalQA.from_chain_type(llm_model, 
                                 chain_type="stuff", 
                                 retriever=retriever)

query = "White Rabbit"
response = qa_chain.run(query)
print(f"{response=}")

# Evaluation of a bunch of queries

In [ ]:
question_answers = [
    {'question' : "Who is the protagonist?", 
     'answer' : "Alice"},
    {'question' : "Who is Alice?", 
     'answer' : 
        """Alice is the curious young girl who goes on a wild
        adventure full of strange and wonderful creatures."""},
    {'question' : "Who is the Caterpilar?", 
     'answer' : 
        """Caterpillar is an old wise sage who has been
        around for hundreds of years. He is a mysterious
        creature with an ever-changing form."""},
    {'question' : "Who is the Chesshire Cat?", 
     'answer' : 
        """The Cheshire Cat is a mysterious and mischievous
        creature who loves to play pranks on Alice. He
        is fond of disappearing and reappearing,"""},
    {'question' : "Who is the Mad Hatter?", 
     'answer' : 
        """The Mad Hatter loves to throw mad tea-parties with
        his friends, the March Hare and the Dormouse. He is
        always dressed in the craziest outfits, and loves
        to talk in riddles and ask nonsensical questions."""},
    {'question' : "Who is the Queen of Hearts?", 
     'answer' : 
        """The Queen of Hearts rules the land with an iron fist. She is
        known for her grandiose parties, where she can often be
        found shouting orders and demanding absolute obedience.
        Her favorite game is croquet, a game she plays with a
        rather unorthodox set of rules that only she understands."""},
    {'question' : "Who is the White Rabbit?", 
     'answer' : 
        """White Rabbit is an anxious, time-obsessed
        character from Alice in Wonderland who is always
        running late and is constantly in a hurry."""},
]
print(f"There are {len(question_answers)} question answers")

In [ ]:
fake_responses = [item['answer'] for item in question_answers]
print(f"There are {len(fake_responses)} fake responses")

In [ ]:
from langchain.chains import RetrievalQA

llm_service = FakeLLMStrategy(fake_responses)
#llm_sergice = OpenAILLMStrategy(llm_parameters)

llm_model = llm_service.get_model()

# Asking theLLM
# the response will be based on the retrieved documents 
qa_chain = RetrievalQA.from_chain_type(llm_model, 
                                 chain_type="stuff", 
                                 retriever=retriever,  
                                 input_key="question")


In [ ]:
from pprint import pprint

predictions = qa_chain.apply(question_answers)
    
print(f"There are {len(predictions)} predictions")
pprint(predictions)

# Evaluate answers

In [ ]:
fake_responses = ['CORRECT' for item in question_answers]
fake_eval = ( retriever_parameters['k'] > 5 and 
        retriever_parameters['score_threshold'] > 0.9 and 
        splitter_parameters['chunk_size'] <= 2000 )           
fake_responses[0] = 'CORRECT' if fake_eval else 'INCORRECT' 
print(f"There are {len(fake_responses)} responses")

In [ ]:
from langchain.evaluation.qa import QAEvalChain


llm_service = FakeLLMStrategy(fake_responses)
#llm_sergice = OpenAILLMStrategy(llm_parameters)

llm_model = llm_service.get_model()

# Start your eval chain
eval_chain = QAEvalChain.from_llm(llm_model)

# Have it grade itself. The code below helps the eval_chain know where the different parts are
graded_outputs = eval_chain.evaluate(question_answers,
                                     predictions,
                                     question_key="question",
                                     prediction_key="result",
                                     answer_key='answer')
graded_outputs

# Retriever optimization

TODO optimize for a list of queries

In [ ]:
!pip install optuna 1>/dev/null

In [ ]:
# https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html

In [ ]:
search_types = ["similarity", "mmr",  "similarity_score_threshold"]

In [ ]:
from functools import partial

# function to optimize
def get_retriever(vectorstore, search_type, k, score_threshold):
    retriever_parameters = {
        'k': k,
        # close to 0 means very similar
        'score_threshold': score_threshold
    }
    search_service = RetrieverSearchStrategy(vectorstore, "mmr", retriever_parameters)
    
    return search_service.get_retriever()

# currying partial
get_retriever_ftom_vectorstore = partial(get_retriever, vectorstore)

In [ ]:
retriever = get_retriever_ftom_vectorstore(search_types[0], 5, 0.3)
# retrieve some indexed documents relevant for this query
query = "White Rabbit"
docs = retriever.get_relevant_documents(query)
print(f"Found {len(docs)}")

In [ ]:
# function scoring
def evaluate_retriever(retriever):
    query = "White Rabbit"
    search_results = retriever.get_relevant_documents(query)

    # how many documents actually contins the query
    assessment = [ 1 if query in result.page_content else 0
                  for result in search_results]
    return sum(assessment)

In [ ]:
score = evaluate_retriever(retriever)
print(f"{score=}")

In [ ]:
import optuna
from optuna.study import StudyDirection

# function to be minimized
def objective(trial):
    search_type = trial.suggest_categorical('search_type', search_types)
    k = trial.suggest_int('k', 1, 20)
    score_threshold = trial.suggest_float('score_threshold', 0.0, 1.0)
    return evaluate_retriever(get_retriever_ftom_vectorstore(search_type, k, score_threshold))

study = optuna.create_study(direction=StudyDirection.MAXIMIZE)
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=100)

study.best_params

# Document search optimization

idem plus splitter configuration

TODO study  name

# QA optimization

idem plus QA Evaluation

In [ ]:
#
#
#

In [ ]:
from math import ceil
from pathlib import Path

# import pytest
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.embeddings import FakeEmbeddings
from langchain.llms.fake import FakeListLLM
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS


# @pytest.mark.xfail
def test_all_in_one():
    # This is the source document.
    data_folder = "tests/unit/data"
    document_path = f"{data_folder}/hr_txt/carpool_policy.txt"

    file = Path(document_path)
    size = file.stat().st_size

    # Setup a text loader.
    loader = TextLoader(document_path)
    documents = loader.load()

    assert len(documents) == 1

    first_document = documents[0].page_content
    assert len(first_document) == size

    # Get your splitter ready.
    chunk_size = 1000
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)

    # Split your docs into texts.
    # assumes the chunk count is size / chunk size
    texts = text_splitter.split_documents(documents)
    nr_chunks = ceil(size / chunk_size)
    assert len(texts) == nr_chunks

    # Get embedding engine ready.
    # embeddings = OpenAIEmbeddings()
    embeddings = FakeEmbeddings(size=1352)

    # Embedd your texts andd store them in the vector database.
    # database is in memory.
    vector_store = FAISS.from_documents(texts, embeddings)
    assert len(vector_store.docstore._dict.keys()) == nr_chunks

    # Init a retriever for this db
    # lookup for some relevqnt parts
    nr_results = 1
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": nr_results})

    # retrieve some indexed documents relevant for this query
    query = "policy"
    docs = retriever.get_relevant_documents(query)
    assert len(docs) == nr_results

    # create a chain to answer questions with this documents
    responses = ["Action: Policy understanding", "Final Answer: answer"]

    qa = RetrievalQA.from_chain_type(
        llm=FakeListLLM(responses=responses), chain_type="stuff", retriever=retriever, return_source_documents=True
    )

    response = qa({"query": query})

    assert response["result"] == responses[0]


# @pytest.mark.xfail
def test_update_vector_store():
    # This are the source documents.
    documents_folder = "tests/unit/data/hr_txt"
    documents_names = ["carpool_policy.txt", "holidays_policy.txt"]

    # Get your splitter ready.
    chunk_size = 1000
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)

    # Get embedding engine ready.
    # embeddings = OpenAIEmbeddings()
    embeddings = FakeEmbeddings(size=1352)

    vector_store = None

    for document_name in documents_names:
        document_path = f"{documents_folder}/{document_name}"
        file = Path(document_path)
        size = file.stat().st_size

        # Setup a text loader.
        loader = TextLoader(document_path)
        documents = loader.load()
        assert len(documents) == 1

        first_document = documents[0].page_content
        assert len(first_document) == size

        # Split your docs into texts.
        # assumes the chunk count is size / chunk size
        texts = text_splitter.split_documents(documents)
        nr_chunks = ceil(size / chunk_size)
        expected_nr_chunks = (nr_chunks, nr_chunks + 1)
        assert len(texts) in expected_nr_chunks

        # Embedd your texts andd store them in the vector database.
        # database is in memory.
        if vector_store:
            nr_before = len(vector_store.docstore._dict.keys())
            # vector_store.add_texts(texts)
            vector_store.add_documents(texts)
            nr_after = nr_chunks + nr_before
            expected_nr_chunks = (nr_after, nr_after + 1, nr_after + 2)
            assert len(vector_store.docstore._dict.keys()) in expected_nr_chunks

        else:
            vector_store = FAISS.from_documents(texts, embeddings)
            assert len(vector_store.docstore._dict.keys()) in expected_nr_chunks


# @pytest.mark.xfail
def test_metadata():
    # This is the source document.
    data_folder = "tests/unit/data"
    document_path = f"{data_folder}/hr_txt/carpool_policy.txt"

    file = Path(document_path)
    size = file.stat().st_size

    # Setup a text loader.
    loader = TextLoader(document_path)
    documents = loader.load()

    assert len(documents) == 1

    first_document = documents[0].page_content
    assert len(first_document) == size

    # set metadata
    for document in documents:
        document.metadata = {"path": document_path}

    # Get your splitter ready.
    chunk_size = 1000
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=100)

    # Split your docs into texts.
    # assumes the chunk count is size / chunk size
    texts = text_splitter.split_documents(documents)
    nr_chunks = ceil(size / chunk_size)
    assert len(texts) == nr_chunks

    first_test = texts[0]
    assert "path" in first_test.metadata
    assert isinstance(first_test, Document)
    assert first_test.metadata["path"] == document_path

    # Get embedding engine ready.
    # embeddings = OpenAIEmbeddings()
    embeddings = FakeEmbeddings(size=1352)

    # Embedd your texts andd store them in the vector database.
    # database is in memory.
    # split actually returns Langchain's Document objects
    # it keeps track of metadata
    vector_store = FAISS.from_documents(texts, embeddings)
    assert len(vector_store.docstore._dict.keys()) == nr_chunks

    # Init a retriever for this db
    # lookup for some relevqnt parts
    nr_results = 1
    retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": nr_results})

    # retrieve some indexed documents relevant for this query
    query = "policy"
    docs = retriever.get_relevant_documents(query)
    assert len(docs) == nr_results

    # create a chain to answer questions with this documents
    responses = ["Action: Policy understanding", "Final Answer: answer"]

    qa = RetrievalQA.from_chain_type(
        llm=FakeListLLM(responses=responses), chain_type="stuff", retriever=retriever, return_source_documents=True
    )

    response = qa({"query": query})

    assert response["result"] == responses[0]
    assert "source_documents" in response.keys()
    assert len(response["source_documents"]) == 1

    first_source = response["source_documents"][0]
    assert isinstance(first_source, Document)
    assert first_source.metadata["path"] == document_path


In [ ]:
    def get_sources_with_similarity_and_score(self, user_question: str) -> Dict[Any, float]:
        results_with_scores = self.vector_store.similarity_search_with_score(
            user_question,
            k=self.llm_context["retriever_k"],
            score_threshold=self.llm_context["retriever_score_threshold"],
        )
        # for doc, score in results_with_scores:
        #    logger.info(f"Content: {doc.page_content}, Metadata: {doc.metadata}, Score: {score}")
        # warning - eats the iterator
        return results_with_scores

    def get_sources_with_mmr_and_score(self, user_question: str) -> List[Any]:
        results = self.vector_store.max_marginal_relevance_search(user_question)
        # for doc in results:
        #    logger.into(f"Content: {doc.page_content}, Metadata: {doc.metadata}")
        # warning - eats the iterator
        return results

    def get_sources_with_retriever(self, user_question: str) -> List[Any]:
        results = self.retriever.get_relevant_documents(user_question)
        # for doc in results:
        #    logger.info(f"Content: {doc.page_content}, Metadata: {doc.metadata}")
        # warning - eats the iterator
        return results
